In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from catboost import CatBoostRegressor

In [3]:
RANDOM_STATE = 12345

CATIGORIES_TRIM = ["base", "extra level touring", "limited", "luxury touring", "level extra", "standard level extra", "standard level touring", "classic edition", "custom edition",
              "deluxe", "deluxe level", "extra", "grand touring", "luxury","luxury edition", "luxury sport", "luxury special", "luxury touring", "sport edition", 
              "special edition", "special equipment","standart level", "standart", "special version", "touring", 
              "xlt", "ltd", "ltz", "gle", "sle", "slt", "ce",  "dx",  "dl",  "ex",  "gl",  "gt", "lx",  "le", "sl", "sv",  "ls", "lt",  "se",]

CATIGORIES_COLUMNS = ["trim", "transmission", "state", "color", "interior", "body", "make", "model"]

In [4]:
data_train = pd.read_csv("./used-cars-price-prediction-19ds/train.csv")
data_test = pd.read_csv("./used-cars-price-prediction-19ds/test.csv")

In [5]:
def make_unique(data):
    if "ford" in data:
        return "ford"
    if "gmc" in data:
        return "gmc"
    if "land" in data:
        return "landrover"
    if "mercedes" in data:
        return "mercedes"
    if data == "vw":
        return "volkswagen"
    if "dodge" in data:
        return "dodge"
    if "mazda" in data:
        return "mazda"
    else:
        return data

In [6]:
def body_unique(data):
    if data.find("cab") != -1 or data == "koup":
        return "pick-up"
    if data.find("convertible") != -1:
        return "convertible"
    if data.find("coupe") != -1:
        return "coupe"
    if data.find("wagon") != -1:
        return "wagon"
    if data.find("van") != -1:
        return "van"
    if data.find("sedan") != -1:
        return "sedan"
    if data.find("van") != -1:
        return "van"
    else:
        return data

In [7]:
def trim_unique(data):

    def classify_trim_levels(trim_levels, category):
        for trim_level in trim_levels:
            if trim_level in category:
                return trim_level
        return "unknown"

    trim = classify_trim_levels(CATIGORIES_TRIM, data)
    if trim == "extra level touring":
        return "xlt"
    if trim == "limited":
        return "ltd"
    if trim == "luxury touring":
        return "ltz"
    if trim == "level extra":
        return "gle"
    if trim == "standard level extra":
        return "sle"
    if trim == "standard level touring":
        return "slt"
    if trim == "classic edition" or trim == "custom edition":
        return "ce"
    if trim == "deluxe":
        return "dx"
    if trim == "deluxe level":
        return "dl"
    if trim == "extra":
        return "ex"
    if trim == "grade level":
        return "gl"
    if trim == "grand touring":
        return "gt"
    if trim == "luxury":
        return "lx"
    if trim == "luxury edition":
        return "le"
    if trim == "luxury sport":
        return "ls"
    if trim == "luxury touring":
        return "lt"
    if trim == "sport edition" or trim == "special edition" or trim =="special equipment":
        return "se"
    if trim == "standart level" or trim == "standart":
        return "sl"
    if trim == "special version":
        return "sv"
    if trim == "touring":
        return "t"
    else:
        return trim

In [8]:
def getfullitemsforOHE(wholedf,featlist,sort=True):
    def sortornot(X):
        if sort==False:
            return X
        else:
            return sorted(X)
       
    fulllist=[]
    for feat in featlist:
        fulllist.append(sortornot(wholedf[feat].unique()))
    return fulllist

In [9]:
def preprocessing(data):
    data["saledate"] = pd.to_datetime(data["saledate"])
    data["saledate"] = pd.to_datetime(data["saledate"], utc=True)

    data["car_age"] = data["saledate"].dt.year - data["year"]

    data = data.drop(["seller", "vin", "saledate"], axis=1)
    data[["color", "interior"]] = data[["color", "interior"]].fillna('—')

    catigories = ["trim", "transmission", "state", "color", "interior", "body", "make", "model"]

    for i in catigories:
        data[i] = data[i].apply(lambda x: str(x).lower())
        data[i] = data[i].fillna("unknown")

    data["body"] = data["body"].apply(body_unique)
    data["make"] = data["make"].apply(make_unique)
    data["trim"] = data["trim"].apply(trim_unique)

    return data

In [10]:
data_train = preprocessing(data_train)
data_test = preprocessing(data_test)

In [11]:
data_train["condition"] = data_train["condition"].apply(lambda x: round(x*2)/2 if pd.isnull(x) is not True else x)
data_test["condition"] = data_test["condition"].apply(lambda x: round(x*2)/2 if pd.isnull(x) is not True else x)

In [12]:
for i in list(data_train.groupby("condition")["odometer"].mean().index):
    data_train.loc[(data_train["odometer"] >= 0.19 * (10 ** 6))&(data_train["condition"] == i), "odometer"] = data_train.groupby("condition")["odometer"].mean()[i]
    data_test.loc[(data_test["odometer"] >= 0.19 * (10 ** 6))&(data_test["condition"] == i), "odometer"] = data_test.groupby("condition")["odometer"].mean()[i]

In [13]:
cats = getfullitemsforOHE(data_train, CATIGORIES_COLUMNS)

In [14]:
data_train = data_train.reset_index(drop=True)

In [15]:
ohe=OneHotEncoder(categories=cats, sparse=False, handle_unknown="ignore", drop='first').fit(data_train[CATIGORIES_COLUMNS])

X_train = ohe.transform(data_train[CATIGORIES_COLUMNS])
ohe_train = pd.DataFrame(X_train,columns=ohe.get_feature_names_out(CATIGORIES_COLUMNS))

X_test = ohe.transform(data_test[CATIGORIES_COLUMNS])
ohe_test = pd.DataFrame(X_test,columns=ohe.get_feature_names_out(CATIGORIES_COLUMNS))

c:\Users\Rog\Desktop\Programing\Python\envs\Kaggle\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\Rog\Desktop\Programing\Python\envs\Kaggle\Lib\site-packages\sklearn\preprocessing\_encoders.py:202: UserWarning: Found unknown categories in columns [6, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [16]:
data_train = data_train.drop(CATIGORIES_COLUMNS, axis=1)
data_train = data_train.join(ohe_train)

data_test = data_test.drop(CATIGORIES_COLUMNS, axis=1)
data_test = data_test.join(ohe_test)

In [17]:
features = data_train.drop(["sellingprice"], axis=1)
target = data_train["sellingprice"]

In [18]:
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.15, random_state=RANDOM_STATE, shuffle=True)

In [19]:
cat = CatBoostRegressor(n_estimators=10000, depth=16, random_state=RANDOM_STATE, silent=True, task_type="GPU", devices="0:1", early_stopping_rounds=2)

In [20]:
cat.fit(features_train, target_train, eval_set=(features_test, target_test), plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [21]:
results = pd.DataFrame(pd.read_csv("./used-cars-price-prediction-19ds/test.csv")["vin"]).join(pd.DataFrame(cat.predict(data_test)))
results.columns = ["vin", "sellingprice"]

In [22]:
results.to_csv("result.csv", index=False)